# DINOv3 vs EUPE — STM 调制区域语义分割对比

**主任务**: 3 类语义分割

| class_id | 类别 |
|----------|------|
| 0 | background |
| 1 | modulation_region |
| 2 | sqrt2_modulation_region |

**Encoder 对比**:
| Encoder | 来源 | embed_dim | depth | 预训练数据 |
|---------|------|-----------|-------|-----------|
| DINOv3 ViT-L | Facebook Research | 1024 | 24 | 自然图像 |
| EUPE ViT-B | 官方 vendor/EUPE | 768 | 12 | 科学图像 |

**数据流**: LabelMe PNG + JSON → resize 512×512 → Encoder → DINOv3LinearSegmentationHead → mask

**标注源**: `data/stm_dataset/FeTe-sxm/png-modulation/*.json` (LabelMe format, polygon 像素坐标)

In [ ]:
from __future__ import annotations

import json, os, random, sys
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as nn_functional
from torch.utils.data import DataLoader, Dataset, Subset
import matplotlib.pyplot as plt
from PIL import Image as PILImage, ImageDraw

# repo root 自动定位
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for parent in [start, *start.parents]:
        if (parent / 'src' / 'lumen').exists():
            return parent
    raise RuntimeError('Could not find repo root')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
if str(REPO_ROOT / 'hyper-data-main' / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'hyper-data-main' / 'src'))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SXM_DIR = REPO_ROOT / 'data' / 'stm_dataset' / 'FeTe-sxm'
MOD_DIR = SXM_DIR / 'png-modulation'   # 调制区域标注 (LabelMe JSON + PNG)
DEF_DIR = SXM_DIR / 'png-defect'       # 缺陷标注 (LabelMe JSON + PNG)

# ── 两个 Encoder 的 checkpoint ──
DINOV3_PATH = REPO_ROOT / 'checkpoints' / 'dinov3-vitl16-pretrain-lvd1689m'
EUPE_PATH   = REPO_ROOT / 'checkpoints' / 'EUPE-ViT-B.pt'

# 类别定义
CLASS_NAMES = ['background', 'modulation_region', 'sqrt2_modulation_region']
NUM_CLASSES = len(CLASS_NAMES)
LABEL_TO_CLASS_ID = {
    'modulation_region': 1,
    'sqrt2_modulation_region': 2,
}

IMAGE_SIZE = 512
BATCH_SIZE = 2
EPOCHS = 80
HEAD_LR = 5e-4          # Head 学习率
ENCODER_LR = 5e-5       # Encoder 微调学习率 (低 10 倍)
WEIGHT_DECAY = 1e-4
DICE_WEIGHT = 0.5       # Dice loss 权重 (0=纯CE, 1=纯Dice)
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'Device: {DEVICE}')
print(f'Repo root: {REPO_ROOT}')
print(f'SXM dir exists:         {SXM_DIR.exists()}')
print(f'MOD_DIR exists:         {MOD_DIR.exists()}')
print(f'DEF_DIR exists:         {DEF_DIR.exists()}')
print(f'DINOv3 backbone exists: {DINOV3_PATH.exists()}')
print(f'EUPE backbone exists:   {EUPE_PATH.exists()}')

## 1. LabelMe 数据集 + 加载

从 `png-modulation/` 读取 LabelMe JSON 标注 + 对应 PNG 图像：
- 每个 `.json` 文件包含 polygon 标注（像素坐标）
- 没有标注的 PNG 也加入作为全背景样本
- `__getitem__` 返回 (1, 512, 512) image + (512, 512) mask

In [ ]:
class LabelMeSegDataset(Dataset):
    """从 LabelMe JSON + 对应 PNG 构建语义分割数据集。"""

    def __init__(
        self,
        data_dir: str | Path,
        label_to_id: dict[str, int],
        image_size: int = 512,
        augment: bool = False,
        include_unlabeled: bool = False,  # 默认不在训练中加入未标注图
    ):
        super().__init__()
        self.data_dir = Path(data_dir)
        self.label_to_id = label_to_id
        self.image_size = image_size
        self.augment = augment

        self.samples: list[dict[str, Any]] = []
        json_files = sorted(self.data_dir.glob('*.json'))

        for jf in json_files:
            png_path = jf.with_suffix('.png')
            if not png_path.exists():
                continue
            with open(jf, encoding='utf-8') as f:
                ann = json.load(f)
            shapes = [s for s in ann.get('shapes', []) if s['label'] in self.label_to_id]
            if not shapes:
                continue
            self.samples.append({
                'stem': jf.stem, 'png_path': png_path,
                'shapes': shapes,
                'img_h': ann.get('imageHeight', 0),
                'img_w': ann.get('imageWidth', 0),
            })

        if include_unlabeled:
            labeled_stems = {s['stem'] for s in self.samples}
            for png_path in sorted(self.data_dir.glob('*.png')):
                if png_path.stem not in labeled_stems:
                    self.samples.append({
                        'stem': png_path.stem, 'png_path': png_path,
                        'shapes': [], 'img_h': 0, 'img_w': 0,
                    })

        self.stems = [s['stem'] for s in self.samples]

    def __len__(self) -> int:
        return len(self.samples)

    def _render_mask(self, shapes: list[dict], h: int, w: int) -> np.ndarray:
        mask = PILImage.new('L', (w, h), color=0)
        drawer = ImageDraw.Draw(mask)
        for shape in shapes:
            cid = self.label_to_id.get(shape['label'])
            if cid is None:
                continue
            pts = shape['points']
            st = shape.get('shape_type', 'polygon')
            if st == 'polygon' and len(pts) >= 3:
                drawer.polygon([(float(p[0]), float(p[1])) for p in pts], fill=cid)
            elif st == 'rectangle' and len(pts) >= 2:
                x0, y0 = float(pts[0][0]), float(pts[0][1])
                x1, y1 = float(pts[1][0]), float(pts[1][1])
                drawer.rectangle([min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1)], fill=cid)
        return np.asarray(mask, dtype=np.int64)

    def _load_and_resize(self, sample: dict) -> tuple[np.ndarray, np.ndarray]:
        img = PILImage.open(sample['png_path']).convert('L')
        W, H = img.size
        mask = self._render_mask(sample['shapes'], H, W)

        S = self.image_size
        img_arr = np.asarray(img.resize((S, S), PILImage.BILINEAR), dtype=np.float32) / 255.0
        mask_pil = PILImage.fromarray(mask.astype(np.int32), mode='I')
        mask_arr = np.asarray(mask_pil.resize((S, S), PILImage.NEAREST), dtype=np.int64)
        return img_arr, mask_arr

    def _maybe_augment(self, image: np.ndarray, mask: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        if not self.augment:
            return image, mask
        if random.random() < 0.5:
            image = image[:, ::-1].copy(); mask = mask[:, ::-1].copy()
        if random.random() < 0.5:
            image = image[::-1, :].copy(); mask = mask[::-1, :].copy()
        k = random.randint(0, 3)
        if k:
            image = np.rot90(image, k).copy(); mask = np.rot90(mask, k).copy()
        if random.random() < 0.5:
            gain = 1.0 + (random.random() - 0.5) * 0.4
            bias = (random.random() - 0.5) * 0.2
            image = np.clip(image * gain + bias, 0.0, 1.0)
        return image, mask

    def __getitem__(self, index: int) -> dict[str, Any]:
        sample = self.samples[index]
        image, mask = self._load_and_resize(sample)
        image, mask = self._maybe_augment(image, mask)
        image_t = torch.from_numpy(image.astype(np.float32)).unsqueeze(0)
        mask_t = torch.from_numpy(mask.astype(np.int64))
        return {'image': image_t, 'mask': mask_t, 'stem': sample['stem']}


# ── 加载调制区域数据集（只用有标注的 4 张） ──
dataset = LabelMeSegDataset(
    data_dir=MOD_DIR,
    label_to_id=LABEL_TO_CLASS_ID,
    image_size=IMAGE_SIZE,
    augment=False,
    include_unlabeled=False,
)

print(f'Classes: {CLASS_NAMES} (num={NUM_CLASSES})')
print(f'Loaded {len(dataset)} labeled samples')
for s in dataset.samples:
    labels = set(sh['label'] for sh in s['shapes'])
    print(f'  {s["stem"]}: {len(s["shapes"])} shapes, labels={labels}')

## 2. 标注对齐验证（人工检查点）

⚠️ **必须人工肉眼确认**：mask 是否正确覆盖调制区域。
如果错位明显，说明 LabelMe 标注坐标有问题。

In [ ]:
# 验证：对有标注的样本画灰度图 + mask 叠加
from matplotlib.lines import Line2D

CLASS_COLORS = {1: 'cyan', 2: 'magenta'}

labeled_indices = [i for i, s in enumerate(dataset.samples) if s['shapes']]
n_show = min(4, len(labeled_indices))
fig, axes = plt.subplots(2, n_show, figsize=(5 * n_show, 10))
if n_show == 1:
    axes = axes[:, np.newaxis]

for col, idx in enumerate(labeled_indices[:n_show]):
    sample = dataset[idx]
    img = sample['image'][0].numpy()
    mask = sample['mask'].numpy()

    axes[0, col].imshow(img, cmap='gray')
    axes[0, col].set_title(f"{sample['stem']}\nOriginal")
    axes[0, col].axis('off')

    # mask 彩色叠加
    mask_rgb = np.zeros((*mask.shape, 3))
    mask_rgb[mask == 1] = [0, 1, 1]   # cyan
    mask_rgb[mask == 2] = [1, 0, 1]   # magenta
    axes[1, col].imshow(img, cmap='gray')
    axes[1, col].imshow(mask_rgb, alpha=0.4)
    axes[1, col].set_title(f'Mask: mod={int((mask==1).sum())}, sqrt2={int((mask==2).sum())}')
    axes[1, col].axis('off')

legend_elements = [
    Line2D([0], [0], color='cyan', linewidth=4, label='modulation_region (cls=1)'),
    Line2D([0], [0], color='magenta', linewidth=4, label='sqrt2_modulation_region (cls=2)'),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=2)
plt.tight_layout()
plt.show()

print("⚠️ 人工检查：彩色叠加应正确覆盖调制区域。")

## 3. 训练/验证划分 + class 权重

20 张样本量小，用 leave-some-out (80/20 = ~15 train / ~4 val) 划分。
计算每类像素频率，用倒频权重缓解背景占绝大多数的问题。


In [ ]:
# ── 4 张标注图 → 3 张训练 / 1 张验证 ──
all_indices = list(range(len(dataset)))
random.Random(SEED).shuffle(all_indices)

n_val = max(1, len(all_indices) // 4)  # 1 val
val_indices = sorted(all_indices[:n_val])
train_indices = sorted(all_indices[n_val:])

val_dataset = Subset(dataset, val_indices)
print(f'Train: {len(train_indices)}, Val: {len(val_indices)}')
print(f'Train stems: {[dataset.stems[i] for i in train_indices]}')
print(f'Val stems:   {[dataset.stems[i] for i in val_indices]}')

# 训练用增强版本
dataset_aug = LabelMeSegDataset(
    data_dir=MOD_DIR,
    label_to_id=LABEL_TO_CLASS_ID,
    image_size=IMAGE_SIZE,
    augment=True,
    include_unlabeled=False,
)
train_dataset = Subset(dataset_aug, train_indices)

# 每类像素频率
class_pixel_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for i in train_indices:
    sample = dataset[i]
    flat = sample['mask'].numpy().ravel()
    for cid in range(NUM_CLASSES):
        class_pixel_counts[cid] += int((flat == cid).sum())
freqs = class_pixel_counts / class_pixel_counts.sum()
inv = 1.0 / np.clip(freqs, 1e-6, None)
class_weights = torch.tensor(inv / inv.sum() * NUM_CLASSES, dtype=torch.float32).to(DEVICE)

print('Per-class pixel counts:')
for cid, name in enumerate(CLASS_NAMES):
    print(f'  {cid} ({name}): {class_pixel_counts[cid]} px, weight={class_weights[cid].item():.4f}')

# 用于可视化的标注索引
labeled_indices_in_ds = list(range(len(dataset)))

## 4. 构建两个对比模型

| Trainer | Encoder | 训练模式 | Loss |
|---------|---------|----------|------|
| DINOv3 | ViT-L | head_only（冻结） | DiceCELoss |
| EUPE | ViT-B | encoder_and_head（微调） | DiceCELoss |

DINOv3 保持冻结作为 baseline，EUPE 解冻验证微调效果。

In [ ]:
from lumen.models import DINOv3Encoder
from lumen.models.eupe import EUPEEncoder
from lumen.training.downstream import SegmentationTrainer

# ── 自定义 CE + Dice 混合 Loss ──
class DiceCELoss(nn.Module):
    """CrossEntropy + Dice loss 混合，对小区域分割更友好。"""
    def __init__(self, ce_weight=None, dice_weight=0.5, smooth=1.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=ce_weight, ignore_index=-1)
        self.dice_weight = dice_weight
        self.smooth = smooth

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        C = logits.shape[1]
        targets_oh = nn_functional.one_hot(targets, C).permute(0, 3, 1, 2).float()
        dice = 0.0
        n = 0
        for c in range(1, C):  # skip background (class 0)
            p_c = probs[:, c]
            t_c = targets_oh[:, c]
            inter = (p_c * t_c).sum()
            union = p_c.sum() + t_c.sum()
            if union > 0:
                dice += 1.0 - (2.0 * inter + self.smooth) / (union + self.smooth)
                n += 1
        dice_loss = dice / max(n, 1)
        return ce_loss + self.dice_weight * dice_loss


# ── 构建 DINOv3 trainer (head_only, frozen encoder) ──
print('Loading DINOv3 encoder...')
dino_encoder = DINOv3Encoder(model_dir=DINOV3_PATH, device=DEVICE, local_files_only=True)

dino_trainer = SegmentationTrainer(
    encoder=dino_encoder,
    num_classes=NUM_CLASSES,
    trainability='head_only',
    segmentation_head_name='dinov3-linear',
    segmentation_loss='ce',
    scheduler_name='cosine',
    scheduler_t_max=EPOCHS,
    lr=HEAD_LR,
    weight_decay=WEIGHT_DECAY,
).to(DEVICE)
dino_trainer.criterion = DiceCELoss(ce_weight=class_weights.detach().cpu(), dice_weight=DICE_WEIGHT).to(DEVICE)
d_trainable = sum(p.numel() for p in dino_trainer.parameters() if p.requires_grad)
print(f'  DINOv3:  embed_dim={dino_encoder.embed_dim}, depth=24, trainable={d_trainable:,} (head_only)')

# ── 构建 EUPE trainer (encoder_and_head, unfrozen) ──
print('Loading EUPE encoder...')
eupe_encoder = EUPEEncoder.from_pretrained(checkpoint_path=EUPE_PATH, device=DEVICE, strict=False)
eupe_encoder.auto_convert_input_channels = True

eupe_trainer = SegmentationTrainer(
    encoder=eupe_encoder,
    num_classes=NUM_CLASSES,
    trainability='encoder_and_head',
    segmentation_head_name='dinov3-linear',
    segmentation_loss='ce',
    scheduler_name='cosine',
    scheduler_t_max=EPOCHS,
    lr=HEAD_LR,
    encoder_lr=ENCODER_LR,
    weight_decay=WEIGHT_DECAY,
).to(DEVICE)
eupe_trainer.criterion = DiceCELoss(ce_weight=class_weights.detach().cpu(), dice_weight=DICE_WEIGHT).to(DEVICE)
e_trainable = sum(p.numel() for p in eupe_trainer.parameters() if p.requires_grad)
e_total = sum(p.numel() for p in eupe_trainer.parameters())
print(f'  EUPE:     embed_dim={eupe_encoder.embed_dim}, depth=12, total={e_total/1e6:.1f}M, trainable={e_trainable/1e6:.2f}M (encoder_and_head)')

print(f'\nLoss: DiceCELoss(dice_weight={DICE_WEIGHT})')
print(f'DINOv3: head_only  lr={HEAD_LR}')
print(f'EUPE:   encoder_and_head  head_lr={HEAD_LR}  encoder_lr={ENCODER_LR}')

## 5. 训练循环（DINOv3 head_only / EUPE encoder_and_head，共用 DiceCELoss）

In [ ]:
def collate(batch):
    images = torch.stack([b['image'] for b in batch], dim=0)
    masks = torch.stack([b['mask'] for b in batch], dim=0)
    stems = [b['stem'] for b in batch]
    return {'image': images, 'mask': masks, 'stem': stems}

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate)

@torch.no_grad()
def compute_mean_iou(logits: torch.Tensor, mask: torch.Tensor, num_classes: int) -> float:
    pred = logits.argmax(dim=1)
    ious = []
    for c in range(num_classes):
        p = pred == c
        g = mask == c
        inter = (p & g).sum().item()
        union = (p | g).sum().item()
        if union > 0:
            ious.append(inter / union)
    return float(np.mean(ious)) if ious else 0.0


def train_one_model(name: str, trainer, encoder):
    """训练一个模型，返回 history dict 和 best checkpoint 信息。"""
    history = {'train_loss': [], 'val_loss': [], 'val_miou': []}
    best_miou = -1.0
    best_epoch = 0

    CKPT_DIR = REPO_ROOT / 'artifacts' / 'modulation_seg_compare'
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    BEST_CKPT = CKPT_DIR / f'best_{name}.pt'

    print(f'\n{"="*50}')
    print(f'Training: {name}')
    print(f'Training for {EPOCHS} epochs...')
    print(f'{"="*50}')

    for epoch in range(EPOCHS):
        trainer.train()
        epoch_losses = []
        for batch in train_loader:
            batch = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}
            log = trainer.train_step(batch)
            epoch_losses.append(log['loss'])
        train_loss = float(np.mean(epoch_losses))

        trainer.eval()
        val_losses, val_mious = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}
                logits = trainer(batch['image'])
                loss = trainer.compute_loss(logits, batch['mask'])
                val_losses.append(float(loss.item()))
                val_mious.append(compute_mean_iou(logits, batch['mask'], NUM_CLASSES))
        val_loss = float(np.mean(val_losses))
        val_miou = float(np.mean(val_mious))

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_miou'].append(val_miou)

        if val_miou > best_miou:
            best_miou = val_miou
            best_epoch = epoch
            torch.save({'head_state_dict': trainer.head.state_dict(),
                        'epoch': epoch, 'val_miou': val_miou},
                       BEST_CKPT)

        if (epoch + 1) % 20 == 0 or epoch < 5:
            marker = '  [BEST]' if val_miou >= best_miou else ''
            print(f'  [{name}] Epoch {epoch+1:3d}/{EPOCHS}: '
                  f'train={train_loss:.3f}, val={val_loss:.3f}, mIoU={val_miou:.3f}{marker}')

    print(f'\n✓ {name} done. Best mIoU: {best_miou:.4f} (epoch {best_epoch})')
    return history, best_miou, best_epoch, BEST_CKPT


# ── 依次训练两个模型 ──
dino_history, dino_best_miou, dino_best_epoch, DINO_BEST = train_one_model(
    'DINOv3_ViT-L', dino_trainer, dino_encoder,
)
eupe_history, eupe_best_miou, eupe_best_epoch, EUPE_BEST = train_one_model(
    'EUPE_ViT-B', eupe_trainer, eupe_encoder,
)

print(f'\n{"="*50}')
print(f'SUMMARY:')
print(f'  DINOv3 ViT-L best mIoU: {dino_best_miou:.4f} (epoch {dino_best_epoch})')
print(f'  EUPE   ViT-B best mIoU: {eupe_best_miou:.4f} (epoch {eupe_best_epoch})')
print(f'  Best: {"EUPE" if eupe_best_miou > dino_best_miou else "DINOv3"} wins!')
print(f'{"="*50}')

## 6. 对比可视化：Loss & mIoU 曲线 + 预测

In [ ]:
# ── 训练曲线对比 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_arr = np.arange(1, EPOCHS + 1)

axes[0].plot(epochs_arr, dino_history['train_loss'], 'C0-', alpha=0.4, label='DINOv3 train')
axes[0].plot(epochs_arr, dino_history['val_loss'], 'C0-', linewidth=2, label='DINOv3 val')
axes[0].plot(epochs_arr, eupe_history['train_loss'], 'C1-', alpha=0.4, label='EUPE train')
axes[0].plot(epochs_arr, eupe_history['val_loss'], 'C1-', linewidth=2, label='EUPE val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('DiceCELoss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_arr, dino_history['val_miou'], 'C0-', linewidth=2, label=f'DINOv3 best={dino_best_miou:.4f}')
axes[1].axhline(y=dino_best_miou, color='C0', linestyle='--', alpha=0.5)
axes[1].plot(epochs_arr, eupe_history['val_miou'], 'C1-', linewidth=2, label=f'EUPE best={eupe_best_miou:.4f}')
axes[1].axhline(y=eupe_best_miou, color='C1', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Mean IoU')
axes[1].set_title('Validation mIoU'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

fig.suptitle('DINOv3 ViT-L vs EUPE ViT-B — Training Curves', fontsize=13)
plt.tight_layout(); plt.show()

# ── 验证集预测对比 ──
dino_ckpt = torch.load(DINO_BEST, map_location=DEVICE, weights_only=True)
dino_trainer.head.load_state_dict(dino_ckpt['head_state_dict'])
dino_trainer.eval()

eupe_ckpt = torch.load(EUPE_BEST, map_location=DEVICE, weights_only=True)
eupe_trainer.head.load_state_dict(eupe_ckpt['head_state_dict'])
eupe_trainer.eval()

n_show = len(val_indices)
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))
if n_show == 1:
    axes = axes.reshape(1, -1)

@torch.no_grad()
def _predict_and_ious(trainer, img_t, gt):
    pred = trainer(img_t).argmax(dim=1)[0].cpu().numpy()
    ious = {}
    for c in range(NUM_CLASSES):
        p_c = pred == c; g_c = gt == c
        inter = (p_c & g_c).sum(); union = (p_c | g_c).sum()
        ious[c] = inter / union if union > 0 else float('nan')
    return pred, ious

with torch.no_grad():
    for row, idx in enumerate(val_indices):
        sample = dataset[idx]
        gt = sample['mask'].numpy()
        img_t = sample['image'].unsqueeze(0).to(DEVICE)
        stem = sample['stem']

        dino_pred, dino_ious = _predict_and_ious(dino_trainer, img_t, gt)
        eupe_pred, eupe_ious = _predict_and_ious(eupe_trainer, img_t, gt)

        axes[row, 0].imshow(sample['image'][0].numpy(), cmap='gray')
        axes[row, 0].set_title(f"{stem}")
        axes[row, 1].imshow(gt, cmap='tab10', vmin=0, vmax=NUM_CLASSES - 1)
        axes[row, 1].set_title('Ground Truth')

        d_valid = [v for v in dino_ious.values() if not np.isnan(v)]
        d_iou_str = ', '.join(f'{CLASS_NAMES[c][:4]}={dino_ious[c]:.2f}' for c in range(NUM_CLASSES) if not np.isnan(dino_ious[c]))
        axes[row, 2].imshow(dino_pred, cmap='tab10', vmin=0, vmax=NUM_CLASSES - 1)
        axes[row, 2].set_title(f'DINOv3 (mIoU={np.mean(d_valid):.3f})\n{d_iou_str}')

        e_valid = [v for v in eupe_ious.values() if not np.isnan(v)]
        e_iou_str = ', '.join(f'{CLASS_NAMES[c][:4]}={eupe_ious[c]:.2f}' for c in range(NUM_CLASSES) if not np.isnan(eupe_ious[c]))
        axes[row, 3].imshow(eupe_pred, cmap='tab10', vmin=0, vmax=NUM_CLASSES - 1)
        axes[row, 3].set_title(f'EUPE (mIoU={np.mean(e_valid):.3f})\n{e_iou_str}')

        for c in range(4):
            axes[row, c].axis('off')

winner = 'EUPE ViT-B' if eupe_best_miou > dino_best_miou else 'DINOv3 ViT-L'
fig.suptitle(f'Validation Prediction — {winner} wins ({max(dino_best_miou, eupe_best_miou):.4f})', fontsize=14)
plt.tight_layout(); plt.show()

# ── 柱状图对比 ──
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['DINOv3 ViT-L', 'EUPE ViT-B'], [dino_best_miou, eupe_best_miou],
       color=['#2196F3', '#4CAF50'], width=0.5)
ax.set_ylabel('Best Validation mIoU'); ax.set_title('Encoder Comparison')
for i, v in enumerate([dino_best_miou, eupe_best_miou]):
    ax.text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold', fontsize=12)
ax.set_ylim(0, max(dino_best_miou, eupe_best_miou) * 1.2)
plt.tight_layout(); plt.show()

## 7. 缺陷计数（后处理）

缺陷（bright/dark defect）**不参与训练**。从 sxm 灰度图直接用经典方法估算：

- **Bright defect**: 局部最大值（peak_local_max），阈值 = 全局 mean + 2*std，最小间距 5 px
- **Dark defect**: 同样，但对 -image 取最大值

为了避免把调制区域内的细节误判为缺陷，**只在背景类预测区（class==0）内**搜索局部极值。


In [ ]:
from scipy import ndimage as ndi
try:
    from skimage.feature import peak_local_max
except ImportError:
    peak_local_max = None
    print('⚠️ skimage 未安装，缺陷计数将用 scipy 兜底')


def count_defects_on_sxm(
    image_norm: np.ndarray,   # (H, W) float32 in [0, 1]
    bg_mask: np.ndarray,      # (H, W) bool
    *,
    min_distance: int = 5,
    abs_thresh_sigma: float = 2.0,
) -> tuple[int, int, np.ndarray, np.ndarray]:
    """返回 (n_bright, n_dark, bright_coords, dark_coords)。"""
    flat = image_norm[bg_mask]
    if flat.size == 0:
        return 0, 0, np.empty((0, 2)), np.empty((0, 2))
    mu, sigma = float(flat.mean()), float(flat.std())
    thr_high = mu + abs_thresh_sigma * sigma
    thr_low = mu - abs_thresh_sigma * sigma

    img_masked = np.where(bg_mask, image_norm, mu)
    img_neg = np.where(bg_mask, -image_norm, -mu)

    if peak_local_max is None:
        local_max = ndi.grey_dilation(img_masked, size=2 * min_distance + 1)
        bright_mask = (img_masked == local_max) & (img_masked > thr_high) & bg_mask
        bright_coords = np.argwhere(bright_mask)
        local_max_neg = ndi.grey_dilation(img_neg, size=2 * min_distance + 1)
        dark_mask = (img_neg == local_max_neg) & (-img_neg < thr_low) & bg_mask
        dark_coords = np.argwhere(dark_mask)
    else:
        bright_coords = peak_local_max(
            img_masked, min_distance=min_distance, threshold_abs=thr_high
        )
        dark_coords = peak_local_max(
            img_neg, min_distance=min_distance, threshold_abs=-thr_low
        )
    return len(bright_coords), len(dark_coords), bright_coords, dark_coords


# 选择最优模型
best_name = 'EUPE' if eupe_best_miou > dino_best_miou else 'DINOv3'
best_trainer = eupe_trainer if best_name == 'EUPE' else dino_trainer
best_trainer.eval()
print(f'Using best model: {best_name} (mIoU={max(dino_best_miou, eupe_best_miou):.4f})')

# 在验证集每张图上跑一次（两个模型对比）
print(f"\n{'stem':12s}  {'DINOv3_mod':10s}  {'EUPE_mod':10s}  {'DINO bright':10s}  {'EUPE bright':10s}")
print('-' * 75)
with torch.no_grad():
    for idx in val_indices:
        sample = dataset[idx]
        img = sample['image'].unsqueeze(0).to(DEVICE)
        img_np = sample['image'][0].numpy()

        dino_pred = dino_trainer(img).argmax(dim=1)[0].cpu().numpy()
        eupe_pred = eupe_trainer(img).argmax(dim=1)[0].cpu().numpy()

        d_b, d_d, _, _ = count_defects_on_sxm(img_np, dino_pred == 0)
        e_b, e_d, _, _ = count_defects_on_sxm(img_np, eupe_pred == 0)

        print(f"{sample['stem']:12s}  {int((dino_pred==1).sum()):10d}  "
              f"{int((eupe_pred==1).sum()):10d}  {d_b:10d}  {e_b:10d}")

## 8. 推理 demo：给一张新 sxm，输出叠加可视化

这是实际使用场景：扫到一张新图，模型告诉你"哪片是调制区域、有多少缺陷"，辅助下次扫描定位。


In [ ]:
# ── 推理 Demo: 两个模型在标注样本上的叠加对比 ──
demo_idx = labeled_indices[0] if labeled_indices else 0
demo_sample = dataset[demo_idx]
demo_stem = demo_sample['stem']
print(f'Demo: {demo_stem}')

dino_trainer.eval(); eupe_trainer.eval()
img_tensor = demo_sample['image'].unsqueeze(0).to(DEVICE)
img_np = demo_sample['image'][0].numpy()

with torch.no_grad():
    dino_pred = dino_trainer(img_tensor).argmax(dim=1)[0].cpu().numpy()
    eupe_pred = eupe_trainer(img_tensor).argmax(dim=1)[0].cpu().numpy()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 第一行: DINOv3
axes[0, 0].imshow(img_np, cmap='gray')
axes[0, 0].set_title(f'{demo_stem} — Input')
axes[0, 1].imshow(dino_pred, cmap='tab10', vmin=0, vmax=NUM_CLASSES - 1)
axes[0, 1].set_title(f'DINOv3 Pred\nmod={int((dino_pred==1).sum())}px, sqrt2={int((dino_pred==2).sum())}px')
axes[0, 2].imshow(img_np, cmap='gray')
d_overlay = np.zeros((*dino_pred.shape, 4))
d_overlay[dino_pred == 1] = [0, 1, 1, 0.35]
d_overlay[dino_pred == 2] = [1, 0, 1, 0.35]
axes[0, 2].imshow(d_overlay)
axes[0, 2].set_title(f'DINOv3 Overlay')

# 第二行: EUPE
axes[1, 0].imshow(img_np, cmap='gray')
axes[1, 0].set_title(f'{demo_stem} — Input')
axes[1, 1].imshow(eupe_pred, cmap='tab10', vmin=0, vmax=NUM_CLASSES - 1)
axes[1, 1].set_title(f'EUPE Pred\nmod={int((eupe_pred==1).sum())}px, sqrt2={int((eupe_pred==2).sum())}px')
axes[1, 2].imshow(img_np, cmap='gray')
e_overlay = np.zeros((*eupe_pred.shape, 4))
e_overlay[eupe_pred == 1] = [0, 1, 1, 0.35]
e_overlay[eupe_pred == 2] = [1, 0, 1, 0.35]
axes[1, 2].imshow(e_overlay)
axes[1, 2].set_title(f'EUPE Overlay')

for ax in axes.flat:
    ax.axis('off')
fig.suptitle(f'DINOv3 ViT-L vs EUPE ViT-B — Inference Demo on {demo_stem}', fontsize=14)
plt.tight_layout(); plt.show()

## 9. 未标注图像推理（全部 16 张）

用训练好的模型在 16 张无标注图上做推理，检查泛化能力。

In [ ]:
# ── 收集未标注的 PNG ──
labeled_stems_all = {s['stem'] for s in dataset.samples}
unlabeled_pngs = sorted([
    p for p in MOD_DIR.glob('*.png') if p.stem not in labeled_stems_all
])
print(f'Unlabeled images: {len(unlabeled_pngs)}')

def load_image(path, size=IMAGE_SIZE):
    img = PILImage.open(path).convert('L')
    img = img.resize((size, size), PILImage.BILINEAR)
    return np.asarray(img, dtype=np.float32) / 255.0

# ── 加载 best heads ──
dino_ckpt = torch.load(DINO_BEST, map_location=DEVICE, weights_only=True)
dino_trainer.head.load_state_dict(dino_ckpt['head_state_dict'])
dino_trainer.eval()

eupe_ckpt = torch.load(EUPE_BEST, map_location=DEVICE, weights_only=True)
eupe_trainer.head.load_state_dict(eupe_ckpt['head_state_dict'])
eupe_trainer.eval()

@torch.no_grad()
def infer(trainer, img_np):
    t = torch.from_numpy(img_np.astype(np.float32)).unsqueeze(0).unsqueeze(0).to(DEVICE)
    return trainer(t).argmax(dim=1)[0].cpu().numpy()

# ── 对全部 16 张推理 ──
results = []
for png_path in unlabeled_pngs:
    img = load_image(png_path)
    d_pred = infer(dino_trainer, img)
    e_pred = infer(eupe_trainer, img)
    results.append((png_path.stem, img, d_pred, e_pred))

# ── 分两页展示（每页 8 张 × 3 列） ──
for page, start in enumerate([0, 8]):
    batch = results[start:start+8]
    n = len(batch)
    fig, axes = plt.subplots(n, 3, figsize=(15, 3.5 * n))
    if n == 1:
        axes = axes.reshape(1, -1)

    for row, (stem, img, dp, ep) in enumerate(batch):
        axes[row, 0].imshow(img, cmap='gray')
        axes[row, 0].set_title(f'{stem}', fontsize=9)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(dp, cmap='tab10', vmin=0, vmax=NUM_CLASSES - 1)
        mod_px = int((dp == 1).sum())
        sq2_px = int((dp == 2).sum())
        axes[row, 1].set_title(f'DINOv3  mod={mod_px} sq2={sq2_px}', fontsize=9)
        axes[row, 1].axis('off')

        axes[row, 2].imshow(ep, cmap='tab10', vmin=0, vmax=NUM_CLASSES - 1)
        mod_px = int((ep == 1).sum())
        sq2_px = int((ep == 2).sum())
        axes[row, 2].set_title(f'EUPE  mod={mod_px} sq2={sq2_px}', fontsize=9)
        axes[row, 2].axis('off')

    fig.suptitle(f'Unlabeled Images — Page {page+1}/2', fontsize=13)
    plt.tight_layout()
    plt.show()

# ── 统计概要 ──
print(f"\n{'stem':12s}  {'DINOv3 mod':>10s}  {'DINOv3 sq2':>10s}  {'EUPE mod':>10s}  {'EUPE sq2':>10s}")
print('-' * 65)
for stem, img, dp, ep in results:
    print(f'{stem:12s}  {int((dp==1).sum()):10d}  {int((dp==2).sum()):10d}  '
          f'{int((ep==1).sum()):10d}  {int((ep==2).sum()):10d}')

## 10. 自定义挑选可视化

修改下面 `PICK_STEMS` 列表选择要查看的图像名（支持 png-modulation 和 png-defect 中所有 PNG）。

In [ ]:
# ═══════════════════════════════════════════════════
# 修改这个列表，填入你想看的图像 stem
PICK_STEMS = ['FeTe_0004', 'FeTe_0016','FeTe_0017','FeTe_0020']  # ← 改这里
# ═══════════════════════════════════════════════════

# 同时支持 png-modulation 和 png-defect 下的 PNG
all_pngs = {p.stem: p for p in MOD_DIR.glob('*.png')}
for p in DEF_DIR.glob('*.png'):
    if p.stem not in all_pngs:
        all_pngs[p.stem] = p

# 确保 heads 已加载
dino_ckpt = torch.load(DINO_BEST, map_location=DEVICE, weights_only=True)
dino_trainer.head.load_state_dict(dino_ckpt['head_state_dict']); dino_trainer.eval()
eupe_ckpt = torch.load(EUPE_BEST, map_location=DEVICE, weights_only=True)
eupe_trainer.head.load_state_dict(eupe_ckpt['head_state_dict']); eupe_trainer.eval()

@torch.no_grad()
def load_and_predict(path):
    img = np.asarray(PILImage.open(path).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE), PILImage.BILINEAR), np.float32) / 255.
    t = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).to(DEVICE)
    return img, dino_trainer(t).argmax(1)[0].cpu().numpy(), eupe_trainer(t).argmax(1)[0].cpu().numpy()

n = len(PICK_STEMS)
fig, axes = plt.subplots(n, 3, figsize=(15, 4.5 * n))
if n == 1:
    axes = axes.reshape(1, -1)

for row, stem in enumerate(PICK_STEMS):
    png_path = all_pngs.get(stem)
    if png_path is None:
        print(f'Warning: {stem} not found'); continue
    has_label = stem in {s['stem'] for s in dataset.samples}
    tag = ' [labeled]' if has_label else ''

    img_np, d_pred, e_pred = load_and_predict(png_path)

    for col, (mask, title) in enumerate([
        (None, f'{stem}{tag}'),
        (d_pred, f'DINOv3\nmod={int((d_pred==1).sum())} sqrt2={int((d_pred==2).sum())}'),
        (e_pred, f'EUPE\nmod={int((e_pred==1).sum())} sqrt2={int((e_pred==2).sum())}'),
    ]):
        axes[row, col].imshow(img_np, cmap='gray')
        if mask is not None:
            axes[row, col].imshow(mask, cmap='tab10', vmin=0, vmax=NUM_CLASSES-1, alpha=0.5)
        axes[row, col].set_title(title, fontsize=10)
        axes[row, col].axis('off')

plt.tight_layout(); plt.show()

# ── 数值统计 ──
print(f"\n{'stem':12s}  {'DINO mod':>10s} {'DINO sqrt2':>10s} {'EUPE mod':>10s} {'EUPE sqrt2':>10s}")
print('-' * 60)
for stem in PICK_STEMS:
    png_path = all_pngs.get(stem)
    if png_path is None: continue
    _, dp, ep = load_and_predict(png_path)
    print(f'{stem:12s}  {int((dp==1).sum()):10d} {int((dp==2).sum()):10d} {int((ep==1).sum()):10d} {int((ep==2).sum()):10d}')